In [1]:
import os
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split 
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
import segmentation_models_pytorch as smp
from torchmetrics.classification import BinaryConfusionMatrix

class Config:
    def __init__(
        self, 
        active_model="segformer", 
        encoder_name="mit_b0", 
        loss_name="focal",
        learning_rate=1e-4, 
        train_batch_size=8, 
        eval_batch_size=8,
        image_dir="./data/train/images",
        mask_dir="./data/train/masks",
        results_dir="./results",
        graphs_dir="./graphs",
        val_split=0.10,                                # 10% fixed validation set
        dataset_fractions=[0.1, 0.25, 0.5, 0.75, 1.0], # Training subsets
        num_classes=1,
        encoder_weights="imagenet",
        train_from_scratch=False,                      # NEW: Toggle to train from scratch
        included_filth_types=None,
        include_clean_data=True,
        epochs=50,
        patience=5
    ):
        self.IMAGE_DIR = image_dir
        self.MASK_DIR = mask_dir
        self.RESULTS_DIR = results_dir
        self.GRAPHS_DIR = graphs_dir
        
        self.VAL_SPLIT = val_split
        self.DATASET_FRACTIONS = dataset_fractions
        self.NUM_CLASSES = num_classes 
        
        self.ACTIVE_MODEL = active_model.lower() 
        self.ENCODER = encoder_name.lower()      
        self.TRAIN_FROM_SCRATCH = train_from_scratch
        # Set weights to None if training from scratch
        self.ENCODER_WEIGHTS = None if train_from_scratch else encoder_weights
        
        self.LOSS_NAME = loss_name.lower()       
        
        self.INCLUDED_FILTH_TYPES = included_filth_types if included_filth_types is not None else ["meat", "vegetables"]
        self.INCLUDE_CLEAN_DATA = include_clean_data 
        
        self.LEARNING_RATE = learning_rate
        self.TRAIN_BATCH_SIZE = train_batch_size
        self.EVAL_BATCH_SIZE = eval_batch_size
        self.EPOCHS = epochs
        self.PATIENCE = patience

    def get_experiment_name(self):
        # Update name to distinguish between scratch and pretrained runs
        weight_status = "scratch" if self.TRAIN_FROM_SCRATCH else "pretrained"
        return f"{self.ACTIVE_MODEL}_{self.ENCODER}_{weight_status}_{self.LOSS_NAME}_learning_curve"


class UnifiedSegmentationDataset(Dataset):
    def __init__(self, filenames, img_dir, mask_dir, preprocessing_fn=None):
        self.filenames = filenames
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.preprocessing_fn = preprocessing_fn

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        image_path = os.path.join(self.img_dir, fname)
        mask_path = os.path.join(self.mask_dir, os.path.splitext(fname)[0] + ".png")

        image_pil = Image.open(image_path).convert("RGB").resize((512, 512), Image.BILINEAR)
        mask_pil = Image.open(mask_path).convert("RGB").resize((512, 512), Image.NEAREST)
        
        image = np.array(image_pil)
        mask_rgb = np.array(mask_pil)
        
        mask_binarized = np.any(mask_rgb > 0, axis=-1).astype(np.float32)

        if self.preprocessing_fn:
            image = self.preprocessing_fn(image)
            if image.shape[-1] == 3:
                image = image.transpose(2, 0, 1)

        image_tensor = torch.tensor(image, dtype=torch.float32)
        mask_tensor = torch.tensor(mask_binarized, dtype=torch.float32).unsqueeze(0) 

        return image_tensor, mask_tensor, fname


class SMPLightningModule(pl.LightningModule):
    def __init__(self, cfg):
        super().__init__()
        self.save_hyperparameters(ignore=['cfg'])
        self.cfg = cfg
        self.lr = cfg.LEARNING_RATE
        
        model_params = dict(
            encoder_name=cfg.ENCODER, 
            encoder_weights=cfg.ENCODER_WEIGHTS, 
            in_channels=3, 
            classes=cfg.NUM_CLASSES
        )
        
        if cfg.ACTIVE_MODEL == "unet":
            self.model = smp.Unet(**model_params)
        elif cfg.ACTIVE_MODEL == "deeplabv3":
            self.model = smp.DeepLabV3(**model_params)
        elif cfg.ACTIVE_MODEL == "segformer":
            self.model = smp.Segformer(**model_params)
        else:
            raise ValueError(f"Unsupported model: {cfg.ACTIVE_MODEL}")

        if cfg.LOSS_NAME == "bce":
            self.criterion = torch.nn.BCEWithLogitsLoss()
        elif cfg.LOSS_NAME == "dice":
            self.criterion = smp.losses.DiceLoss(mode=smp.losses.BINARY_MODE, from_logits=True)
        elif cfg.LOSS_NAME == "focal":
            self.criterion = smp.losses.FocalLoss(mode=smp.losses.BINARY_MODE)
        elif cfg.LOSS_NAME == "jaccard":
            self.criterion = smp.losses.JaccardLoss(mode=smp.losses.BINARY_MODE, from_logits=True)
        else:
            raise ValueError(f"Unsupported loss function: {cfg.LOSS_NAME}")
        
        self.conf_matrix_metric = BinaryConfusionMatrix()
        
        self.training_step_losses = []
        self.train_epoch_losses = []
        
        self.validation_step_losses = []
        self.val_epoch_losses = []
        self.val_epoch_ious = []
        self.val_epoch_accuracies = [] 
        self.val_epoch_confusion_matrices = []

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, masks, _ = batch
        logits = self(images)
        loss = self.criterion(logits, masks)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.training_step_losses.append(loss.item())
        return loss

    def on_train_epoch_end(self):
        if self.training_step_losses:
            avg_loss = sum(self.training_step_losses) / len(self.training_step_losses)
            self.train_epoch_losses.append(avg_loss)
            self.training_step_losses.clear()

    def validation_step(self, batch, batch_idx):
        images, masks, _ = batch
        logits = self(images)
        loss = self.criterion(logits, masks)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.validation_step_losses.append(loss.item())
        
        preds = (logits > 0).float() 
        self.conf_matrix_metric.update(preds.flatten(), masks.flatten().long())
        return loss

    def on_validation_epoch_end(self):
        if self.validation_step_losses:
            avg_loss = sum(self.validation_step_losses) / len(self.validation_step_losses)
            self.val_epoch_losses.append(avg_loss)
            self.validation_step_losses.clear()
        
        conf_mat = self.conf_matrix_metric.compute()
        tn, fp, fn, tp = conf_mat[0,0].item(), conf_mat[0,1].item(), conf_mat[1,0].item(), conf_mat[1,1].item()
        
        union = tp + fp + fn
        iou = tp / union if union > 0 else 0.0
        
        total_pixels = tn + fp + fn + tp
        accuracy = (tn + tp) / total_pixels if total_pixels > 0 else 0.0
        
        self.val_epoch_ious.append(iou)
        self.val_epoch_accuracies.append(accuracy)
        self.val_epoch_confusion_matrices.append(conf_mat.cpu().numpy().tolist())
        
        self.log("val_iou", iou, prog_bar=True)
        self.log("val_acc", accuracy, prog_bar=True)
        self.conf_matrix_metric.reset()

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)


class SegmentationExperiment:
    def __init__(self, config):
        self.cfg = config
        self.all_filenames = []
        self.strat_labels = [] 
        self._prepare_dataset()
        
        # Safe fallback preprocessing for from-scratch training
        if self.cfg.ENCODER_WEIGHTS is not None:
            self.preprocessing_fn = smp.encoders.get_preprocessing_fn(self.cfg.ENCODER, self.cfg.ENCODER_WEIGHTS)
        else:
            # If no weights are used, simply scale pixel values between 0.0 and 1.0
            self.preprocessing_fn = lambda x: (x / 255.0).astype(np.float32)
            
        os.makedirs(self.cfg.RESULTS_DIR, exist_ok=True)

    def _prepare_dataset(self):
        raw_files = sorted([f for f in os.listdir(self.cfg.IMAGE_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
        for fname in raw_files:
            category = fname.split('-')[0].lower()
            if category == "clean" and self.cfg.INCLUDE_CLEAN_DATA:
                self.all_filenames.append(fname)
                self.strat_labels.append("clean")
            elif category in self.cfg.INCLUDED_FILTH_TYPES:
                self.all_filenames.append(fname)
                self.strat_labels.append(category)

    def run_fractional_holdout(self):
        X_all = np.array(self.all_filenames)
        y_all = np.array(self.strat_labels)
        
        # 1. Lock in the fixed Validation Set first
        X_train_full, X_val, y_train_full, y_val = train_test_split(
            X_all, y_all, 
            test_size=self.cfg.VAL_SPLIT, 
            stratify=y_all, 
            random_state=42
        )
        
        print(f"\n{'='*60}\n🔒 FIXED VALIDATION SET LOCKED: {len(X_val)} samples\n{'='*60}")
        
        val_dataset = UnifiedSegmentationDataset(X_val.tolist(), self.cfg.IMAGE_DIR, self.cfg.MASK_DIR, self.preprocessing_fn)
        val_loader = DataLoader(val_dataset, batch_size=self.cfg.EVAL_BATCH_SIZE, shuffle=False, num_workers=0)
        
        experiment_results = {"config": vars(self.cfg), "fractions": {}}

        # 2. Iterate over the training dataset subsets
        for fraction in self.cfg.DATASET_FRACTIONS:
            fraction_str = f"{int(fraction * 100)}%"
            print(f"\n📉 TRAINING ON {fraction_str} OF AVAILABLE DATA | {self.cfg.get_experiment_name()}")
            
            if fraction < 1.0:
                X_train_sub, _, y_train_sub, _ = train_test_split(
                    X_train_full, y_train_full, 
                    train_size=fraction, 
                    stratify=y_train_full, 
                    random_state=42
                )
            else:
                X_train_sub, y_train_sub = X_train_full, y_train_full
                
            print(f"Training Samples: {len(X_train_sub)} | Validation Samples: {len(X_val)}")
            
            train_dataset = UnifiedSegmentationDataset(X_train_sub.tolist(), self.cfg.IMAGE_DIR, self.cfg.MASK_DIR, self.preprocessing_fn)
            train_loader = DataLoader(train_dataset, batch_size=self.cfg.TRAIN_BATCH_SIZE, shuffle=True, num_workers=0)

            model = SMPLightningModule(self.cfg)

            checkpoint_dir = f"./final_models/{self.cfg.get_experiment_name()}/{fraction_str}"
            checkpoint_callback = ModelCheckpoint(
                dirpath=checkpoint_dir,
                filename="best-model",
                save_top_k=1,
                monitor="val_loss",
                mode="min"
            )
            early_stop_callback = EarlyStopping(monitor="val_loss", patience=self.cfg.PATIENCE, mode="min")

            trainer = pl.Trainer(
                max_epochs=self.cfg.EPOCHS,
                accelerator="gpu" if torch.cuda.is_available() else "cpu",
                devices=1,
                callbacks=[early_stop_callback, checkpoint_callback],
                enable_progress_bar=True,
                logger=False,
                num_sanity_val_steps=0 
            )

            trainer.fit(model, train_loader, val_loader)
            
            best_epoch_idx = np.argmax(model.val_epoch_ious)
            
            experiment_results["fractions"][fraction_str] = {
                "train_loss_history": model.train_epoch_losses,
                "val_loss_history": model.val_epoch_losses,
                "best_val_iou": model.val_epoch_ious[best_epoch_idx],
                "best_val_accuracy": model.val_epoch_accuracies[best_epoch_idx],
                "best_epoch": int(best_epoch_idx + 1),
                "best_confusion_matrix": model.val_epoch_confusion_matrices[best_epoch_idx]
            }

        result_path = os.path.join(self.cfg.RESULTS_DIR, f"{self.cfg.get_experiment_name()}_learning_curve_metrics.json")
        with open(result_path, 'w') as f:
            json.dump(experiment_results, f, indent=4)
            
        print(f"\n✅ All Fractional runs complete! Metrics saved to: {result_path}")


class ExperimentReporter:
    def __init__(self, results_dir="./results", graphs_dir="./graphs"):
        self.results_dir = results_dir
        self.graphs_dir = graphs_dir
        os.makedirs(self.graphs_dir, exist_ok=True)

    def load_experiment_data(self, cfg):
        experiment_name = cfg.get_experiment_name()
        file_path = os.path.join(self.results_dir, f"{experiment_name}_learning_curve_metrics.json")
        with open(file_path, 'r') as f:
            return json.load(f)

    def plot_learning_curve(self, cfg):
        data = self.load_experiment_data(cfg)
        experiment_name = cfg.get_experiment_name()
        
        fractions_str = []
        fractions_num = []
        ious = []
        accs = []

        for frac_str, run_data in data["fractions"].items():
            fractions_str.append(frac_str)
            fractions_num.append(int(frac_str.replace('%', '')))
            
            ious.append(run_data["best_val_iou"])
            accs.append(run_data["best_val_accuracy"])

        fig, ax = plt.subplots(figsize=(8, 6))

        ax.plot(fractions_num, ious, marker='o', label='Validation IoU', color='green', linewidth=2)
        ax.plot(fractions_num, accs, marker='s', label='Validation Accuracy', color='purple', linewidth=2, linestyle='--')

        ax.set_title(f"{experiment_name.upper()}\nLearning Curve across Dataset Sizes (Holdout)")
        ax.set_xlabel("Percentage of Total Training Dataset Used (%)")
        ax.set_ylabel("Metric Score")
        ax.set_xticks(fractions_num)
        ax.set_xticklabels(fractions_str)
        ax.legend(loc='lower right')
        ax.grid(True, linestyle='--', alpha=0.7)

        plt.tight_layout()
        save_path = os.path.join(self.graphs_dir, f"{experiment_name}_learning_curve.png")
        plt.savefig(save_path, bbox_inches='tight')
        plt.close()
        
        print(f"📈 Learning Curve graph saved to: {save_path}")

    def plot_fractional_confusion_matrices(self, cfg):
        data = self.load_experiment_data(cfg)
        experiment_name = cfg.get_experiment_name()
        
        for frac_str, run_data in data["fractions"].items():
            if "best_confusion_matrix" not in run_data:
                continue

            conf_mat = np.array(run_data["best_confusion_matrix"])
            conf_mat_pct = (conf_mat / np.sum(conf_mat)) * 100

            plt.figure(figsize=(6, 5))
            labels = ["Background (0)", "Filth (1)"]
            
            sns.heatmap(conf_mat_pct, annot=True, fmt=".2f", cmap="Greens", xticklabels=labels, yticklabels=labels)
            for t in plt.gca().texts: t.set_text(t.get_text() + " %")
            
            best_epoch = run_data["best_epoch"]
            best_iou = run_data["best_val_iou"]
            
            plt.title(f"{experiment_name.upper()} - {frac_str} Training Data\nNormalized CM (Epoch: {best_epoch} | IoU: {best_iou*100:.1f}%)")
            plt.ylabel("True Label")
            plt.xlabel("Predicted Label")

            save_path = os.path.join(self.graphs_dir, f"{experiment_name}_{frac_str}_confusion_matrix.png")
            plt.savefig(save_path, bbox_inches='tight')
            plt.close()
            print(f"📊 Confusion Matrix for {frac_str} saved to: {save_path}")

In [2]:

cfg = Config( 
    active_model="segformer",     
    encoder_name="mit_b0",        
    loss_name="focal",            
    dataset_fractions=[0.1, 0.25, 0.5, 0.75, 1.0], 
    epochs=40,                    
    train_batch_size=8,
    eval_batch_size=8,
    train_from_scratch=True
)
    
experiment = SegmentationExperiment(cfg)
    

experiment.run_fractional_holdout()
    
reporter = ExperimentReporter(results_dir=cfg.RESULTS_DIR, graphs_dir=cfg.GRAPHS_DIR)
reporter.plot_learning_curve(cfg)
reporter.plot_fractional_confusion_matrices(cfg)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 3070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



🔒 FIXED VALIDATION SET LOCKED: 68 samples

📉 TRAINING ON 10% OF AVAILABLE DATA | segformer_mit_b0_scratch_focal_learning_curve
Training Samples: 60 | Validation Samples: 68


┏━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name               ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model              │ Segformer             │  3.7 M │ train │     0 │
│ 1 │ criterion          │ FocalLoss             │      0 │ train │     0 │
│ 2 │ conf_matrix_metric │ BinaryConfusionMatrix │      0 │ train │     0 │
└───┴────────────────────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 3.7 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.7 M                                                                                                
Total estimated model params size (MB): 14                                                                         
Modules in train mode: 195                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...


📉 TRAINING ON 100% OF AVAILABLE DATA | segformer_mit_b0_scratch_focal_learning_curve
Training Samples: 607 | Validation Samples: 68


┏━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name               ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model              │ Segformer             │  3.7 M │ train │     0 │
│ 1 │ criterion          │ FocalLoss             │      0 │ train │     0 │
│ 2 │ conf_matrix_metric │ BinaryConfusionMatrix │      0 │ train │     0 │
└───┴────────────────────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 3.7 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.7 M                                                                                                
Total estimated model params size (MB): 14                                                                         
Modules in train mode: 195                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()


✅ All Fractional runs complete! Metrics saved to: ./results/segformer_mit_b0_scratch_focal_learning_curve_learning_curve_metrics.json
📈 Learning Curve graph saved to: ./graphs/segformer_mit_b0_scratch_focal_learning_curve_learning_curve.png
📊 Confusion Matrix for 10% saved to: ./graphs/segformer_mit_b0_scratch_focal_learning_curve_10%_confusion_matrix.png
📊 Confusion Matrix for 25% saved to: ./graphs/segformer_mit_b0_scratch_focal_learning_curve_25%_confusion_matrix.png
📊 Confusion Matrix for 50% saved to: ./graphs/segformer_mit_b0_scratch_focal_learning_curve_50%_confusion_matrix.png
📊 Confusion Matrix for 75% saved to: ./graphs/segformer_mit_b0_scratch_focal_learning_curve_75%_confusion_matrix.png
📊 Confusion Matrix for 100% saved to: ./graphs/segformer_mit_b0_scratch_focal_learning_curve_100%_confusion_matrix.png
